# Wanderbricks Bookings

**Dataset:** `samples.wanderbricks.bookings`

**Difficulty:** Easy

**Topics:** filter, aggregation, date, groupBy

In [0]:
from pyspark.sql import functions as F, types as T

## Learn — Aggregations with Conditions

| Function | What it does |
|----------|-------------|
| `F.sum("col")` | Total sum of a numeric column |
| `F.avg("col")` | Mean value |
| `F.min("col")` / `F.max("col")` | Minimum / maximum value |
| `F.when(cond, val).otherwise(0)` | Conditional value — useful inside `sum()` for conditional counts |
| `F.datediff(F.col("end"), F.col("start"))` | Duration in days between two date columns |

**Docs:** [PySpark Functions](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html) · [PySpark Functions](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html)

In [0]:
# Run this example first — then solve the problems below.
# NOTE: this example is not a solution to any problem

df = spark.table("samples.wanderbricks.bookings")

# Explore schema
df.printSchema()

# Simple aggregation example — total and average booking value
"""
df.agg(
    F.count("*").alias("total_bookings"),
    F.round(F.avg("total_price"), 2).alias("avg_price"),
    F.round(F.sum("total_price"), 2).alias("total_revenue")
).show()
"""

## Problem 1

Count the number of bookings for each **status** value.
Load `samples.wanderbricks.bookings` and group by the `status` column.

**Expected output columns:**
- `status` - booking status (e.g., confirmed, cancelled, pending)
- `booking_count` - number of bookings with that status

In [0]:
# Problem 1 - write your solution here
# Assign your result to: result_1
result_1 = df.groupBy("status").agg(F.count("booking_id").alias("booking_count"))

display(result_1) # replace this

In [0]:
# ── Tests for Problem 1 ──────────────────────────────────────────
assert result_1 is not None, "result_1 is None - did you forget to assign your DataFrame?"
assert hasattr(result_1, 'columns'), "result_1 must be a Spark DataFrame"
cols = [c.lower() for c in result_1.columns]
assert 'status' in cols, "Missing column: status"
assert 'booking_count' in cols, "Missing column: booking_count"
cnt = result_1.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
counts = [r['booking_count'] for r in result_1.collect()]
assert all(c > 0 for c in counts), "All booking_count values must be positive"
print(f"Problem 1 passed ✓  ({cnt} rows)")

## Problem 2

Calculate **total revenue and average booking amount** from confirmed bookings
only (where `status = 'confirmed'`). Return a single-row summary.

**Expected output columns:**
- `total_revenue` - sum of `total_amount` for confirmed bookings
- `avg_booking_amount` - average `total_amount` for confirmed bookings

In [0]:
# Problem 2 - write your solution here
# Assign your result to: result_2
result_2 = df.filter(F.col("status") == "confirmed").agg(F.sum("total_amount").alias("total_revenue"),F.avg("total_amount").alias("avg_booking_amount")).select("total_revenue","avg_booking_amount")
display(result_2)  # replace this

In [0]:
# ── Tests for Problem 2 ──────────────────────────────────────────
assert result_2 is not None, "result_2 is None - did you forget to assign your DataFrame?"
assert hasattr(result_2, 'columns'), "result_2 must be a Spark DataFrame"
cols = [c.lower() for c in result_2.columns]
assert 'total_revenue' in cols, "Missing column: total_revenue"
assert 'avg_booking_amount' in cols, "Missing column: avg_booking_amount"
cnt = result_2.count()
assert cnt == 1, f"Expected exactly 1 row, got {cnt}"
row = result_2.collect()[0]
assert float(row['total_revenue']) > 0, "total_revenue must be > 0 for confirmed bookings"
assert float(row['avg_booking_amount']) > 0, "avg_booking_amount must be > 0"
print(f"Problem 2 passed ✓  ({cnt} rows)")

## Problem 3

Find all bookings that accommodated **more than 4 guests** (`guests_count > 4`).
Return all columns, sorted so the largest groups appear first.

**Expected output columns:**
- All original columns from `samples.wanderbricks.bookings`
- Sorted by `guests_count` descending

In [0]:
# Problem 3 - write your solution here
# Assign your result to: result_3
result_3 = df.filter(F.col("guests_count") > 4).orderBy(F.col("guests_count").desc())

display(result_3)  # replace this

In [0]:
# ── Tests for Problem 3 ──────────────────────────────────────────
assert result_3 is not None, "result_3 is None - did you forget to assign your DataFrame?"
assert hasattr(result_3, 'columns'), "result_3 must be a Spark DataFrame"
cols = [c.lower() for c in result_3.columns]
assert 'guests_count' in cols, "Missing column: guests_count"
assert 'booking_id' in cols, "Missing column: booking_id"
cnt = result_3.count()
assert cnt >= 0, f"Expected rows >= 0, got {cnt}"
if cnt > 0:
    min_guests = result_3.agg(F.min('guests_count')).collect()[0][0]
    assert min_guests > 4, f"All guests_count must be > 4, found min={min_guests}"
    guest_counts = [r['guests_count'] for r in result_3.collect()]
    assert guest_counts == sorted(guest_counts, reverse=True), "Results must be sorted by guests_count descending"
print(f"Problem 3 passed ✓  ({cnt} rows)")

## Problem 4

Count bookings per **month** by extracting year and month from `check_in`.
Use `F.year()` and `F.month()` functions on the date column.

**Expected output columns:**
- `year` - calendar year of the check-in date
- `month` - calendar month (1–12) of the check-in date
- `booking_count` - number of bookings checking in during that month

In [0]:
# Problem 4 - write your solution here
# Assign your result to: result_4 
result_4 = df.groupBy(
    F.year("check_in").alias("year"),
    F.month("check_in").alias("month")
).agg(
    F.count("booking_id").alias("booking_count")
)

display(result_4) # replace this

In [0]:
# ── Tests for Problem 4 ──────────────────────────────────────────
assert result_4 is not None, "result_4 is None - did you forget to assign your DataFrame?"
assert hasattr(result_4, 'columns'), "result_4 must be a Spark DataFrame"
cols = [c.lower() for c in result_4.columns]
assert 'year' in cols, "Missing column: year"
assert 'month' in cols, "Missing column: month"
assert 'booking_count' in cols, "Missing column: booking_count"
cnt = result_4.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
months = [r['month'] for r in result_4.collect()]
assert all(1 <= m <= 12 for m in months), "All month values must be between 1 and 12"
assert all(r['booking_count'] > 0 for r in result_4.collect()), "All booking counts must be positive"
print(f"Problem 4 passed ✓  ({cnt} rows)")

## Problem 5

Calculate the **average stay duration in nights** for each booking status.
Compute the difference between `check_out` and `check_in` (in days) using
`F.datediff()`, then average that per status.

**Expected output columns:**
- `status` - booking status
- `avg_nights` - average number of nights stayed for that status

In [0]:
# Problem 5 - write your solution here
# Assign your result to: result_5
result_5 = df.groupBy("status").agg(F.avg(F.datediff("check_out", "check_in")).alias("avg_nights"))
display(result_5)  # replace this

In [0]:
# ── Tests for Problem 5 ──────────────────────────────────────────
assert result_5 is not None, "result_5 is None - did you forget to assign your DataFrame?"
assert hasattr(result_5, 'columns'), "result_5 must be a Spark DataFrame"
cols = [c.lower() for c in result_5.columns]
assert 'status' in cols, "Missing column: status"
assert 'avg_nights' in cols, "Missing column: avg_nights"
cnt = result_5.count()
assert cnt > 0, f"Expected rows > 0, got {cnt}"
rows = result_5.collect()
assert all(float(r['avg_nights']) > 0 for r in rows), "All avg_nights values must be positive"
assert all(float(r['avg_nights']) < 365 for r in rows), "avg_nights should be < 365 for all statuses"
print(f"Problem 5 passed ✓  ({cnt} rows)")